In [1]:
import re
from pathlib import Path
from email import policy
from email.parser import BytesParser
import numpy as np

def load_email_file(path: Path):
    with open(path, "rb") as f:
        return BytesParser(policy=policy.default).parse(f)

def load_emails_from_folder(folder: Path):
    emails = []
    for p in folder.iterdir():
        if p.is_file():
            try:
                emails.append(load_email_file(p))
            except Exception:
                pass
    return emails

def load_spamassassin_dataset(data_root="data"):
    root = Path(data_root)

    ham_dirs = [root/"easy_ham", root/"easy_ham_2", root/"hard_ham"]
    spam_dirs = [root/"spam", root/"spam_2"]

    for d in ham_dirs + spam_dirs:
        if not d.exists():
            raise FileNotFoundError(f"Missing folder: {d}")

    ham = []
    for d in ham_dirs:
        ham.extend(load_emails_from_folder(d))

    spam = []
    for d in spam_dirs:
        spam.extend(load_emails_from_folder(d))

    X = spam + ham
    y = np.array([1]*len(spam) + [0]*len(ham), dtype=np.int32)  # 1=spam, 0=ham

    return X, y

X, y = load_spamassassin_dataset("data")
print("Total:", len(X), "Spam:", int(y.sum()), "Ham:", int((y==0).sum()))


Total: 9354 Spam: 2400 Ham: 6954


In [2]:
from sklearn.base import BaseEstimator, TransformerMixin

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
NUM_RE = re.compile(r"\b\d+(\.\d+)?\b")

def email_to_text(msg):
    parts = []
    if msg.is_multipart():
        for part in msg.walk():
            ctype = part.get_content_type()
            disp = str(part.get("Content-Disposition", "")).lower()
            if ctype == "text/plain" and "attachment" not in disp:
                try:
                    parts.append(part.get_content())
                except Exception:
                    pass
    else:
        try:
            parts.append(msg.get_content())
        except Exception:
            pass
    return "\n".join(parts)

class EmailToCleanText(BaseEstimator, TransformerMixin):
    def __init__(self, lowercase=True, remove_punct=True, replace_urls=True, replace_numbers=True):
        self.lowercase = lowercase
        self.remove_punct = remove_punct
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        out = []
        for msg in X:
            text = email_to_text(msg)

            if self.lowercase:
                text = text.lower()
            if self.replace_urls:
                text = URL_RE.sub(" URL ", text)
            if self.replace_numbers:
                text = NUM_RE.sub(" NUMBER ", text)
            if self.remove_punct:
                text = re.sub(r"[^\w\s]", " ", text)

            text = re.sub(r"\s+", " ", text).strip()
            out.append(text)
        return out


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", len(X_train), "Test:", len(X_test))


Train: 7483 Test: 1871


train and test output

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

def run_model(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print("\n" + "="*80)
    print(name)
    print("="*80)
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print("\nReport:\n", classification_report(y_test, pred, digits=4))

vectorizer = CountVectorizer(
    binary=True,     
    min_df=2,
    max_df=0.95
)

base = Pipeline([
    ("clean", EmailToCleanText()),
    ("vec", vectorizer),
])

models = {
    "Logistic Regression": Pipeline([("prep", base), ("clf", LogisticRegression(max_iter=2000))]),
    "MultinomialNB":       Pipeline([("prep", base), ("clf", MultinomialNB())]),
    "LinearSVC":           Pipeline([("prep", base), ("clf", LinearSVC())]),
}

for name, model in models.items():
    run_model(name, model)



Logistic Regression
Confusion matrix:
 [[1386    5]
 [  10  470]]

Report:
               precision    recall  f1-score   support

           0     0.9928    0.9964    0.9946      1391
           1     0.9895    0.9792    0.9843       480

    accuracy                         0.9920      1871
   macro avg     0.9912    0.9878    0.9895      1871
weighted avg     0.9920    0.9920    0.9920      1871


MultinomialNB
Confusion matrix:
 [[1383    8]
 [  59  421]]

Report:
               precision    recall  f1-score   support

           0     0.9591    0.9942    0.9764      1391
           1     0.9814    0.8771    0.9263       480

    accuracy                         0.9642      1871
   macro avg     0.9702    0.9357    0.9513      1871
weighted avg     0.9648    0.9642    0.9635      1871



c:\Users\longt\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\longt\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



LinearSVC
Confusion matrix:
 [[1384    7]
 [  13  467]]

Report:
               precision    recall  f1-score   support

           0     0.9907    0.9950    0.9928      1391
           1     0.9852    0.9729    0.9790       480

    accuracy                         0.9893      1871
   macro avg     0.9880    0.9839    0.9859      1871
weighted avg     0.9893    0.9893    0.9893      1871



above are outputs from the TEST set

From the three model outputs, Logistic Regression has the lowest False Positives and False Negatives compated to the other 2 models, so they have the highest precion and recall out of the bunch. making it the most effective classifer amoung the 3

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

best = models["Logistic Regression"]
best.fit(X_train, y_train)

scores = best.predict_proba(X_test)[:, 1]  # spam probability

prec, rec, thresh = precision_recall_curve(y_test, scores)
ap = average_precision_score(y_test, scores)
print("Average Precision (PR AUC-ish):", ap)

target_precision = 0.95
idx = np.where(prec >= target_precision)[0]
if len(idx) == 0:
    print("No threshold reaches that precision. Lower target_precision.")
else:
    i = idx[0] 
    chosen = thresh[i-1] if i > 0 else thresh[0]
    print("Chosen threshold:", chosen)
    print("Precision:", prec[i], "Recall:", rec[i])

    pred = (scores >= chosen).astype(int)
    print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))
    print("\nReport:\n", classification_report(y_test, pred, digits=4))


Average Precision (PR AUC-ish): 0.9981317930639548
Chosen threshold: 0.10730617513785863
Precision: 0.950199203187251 Recall: 0.99375

Confusion matrix:
 [[1365   26]
 [   3  477]]

Report:
               precision    recall  f1-score   support

           0     0.9978    0.9813    0.9895      1391
           1     0.9483    0.9938    0.9705       480

    accuracy                         0.9845      1871
   macro avg     0.9731    0.9875    0.9800      1871
weighted avg     0.9851    0.9845    0.9846      1871

